In [ ]:
%pip install -q torch torchvision tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, ConcatDataset
from tqdm import tqdm
import time

# ------------------------------
# 1️⃣ Dataset & Augmentation
# ------------------------------
DATA_DIR = "/content/drive/MyDrive/twinTnfr/images"

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    normalize,
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

# Original dataset + augmentation duplication to increase size
original_dataset = datasets.ImageFolder(root=DATA_DIR, transform=test_transform)
augmented_dataset = datasets.ImageFolder(root=DATA_DIR, transform=augment_transform)
full_dataset = ConcatDataset([original_dataset, augmented_dataset, augmented_dataset])

train_ratio, val_ratio = 0.7, 0.15
train_size = int(train_ratio * len(full_dataset))
val_size = int(val_ratio * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

train_dataset.dataset.transform = augment_transform
val_dataset.dataset.transform = test_transform
test_dataset.dataset.transform = test_transform

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

classes = original_dataset.classes
print(f"📸 Classes: {classes}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# ------------------------------
# 2️⃣ Model Setup
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pretrained ViT-B/16
model = models.vit_b_16(weights="IMAGENET1K_V1")
model.heads.head = nn.Linear(model.heads.head.in_features, len(classes))
model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

# Use OneCycleLR for faster convergence
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4,
                                          steps_per_epoch=len(train_loader), epochs=10)

scaler = torch.cuda.amp.GradScaler()  # Mixed precision

# ------------------------------
# 3️⃣ Training Utilities
# ------------------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc="Training", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        _, preds = outputs.max(1)
        total_loss += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item(), acc=100. * correct / total)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


📸 Classes: ['Astrocitoma T2', 'Carcinoma T2', 'Ependimoma T2', 'Ganglioglioma T2', 'Glioblastoma T2', 'Granuloma T2', 'Normal T2', 'Oligodendroglioma T2', 'Tuberculoma T2']
Train: 1608, Val: 344, Test: 346
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:06<00:00, 55.8MB/s]
/tmp/ipython-input-1626056975.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision


In [ ]:
# ------------------------------
# Training Function (Reusable)
# ------------------------------
def train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device, start_epoch, end_epoch, save_path="best_vit_model.pth"):
    best_val_acc = 0.0
    for epoch in range(start_epoch, end_epoch):
        print(f"\n🌀 Epoch {epoch+1}/{end_epoch}")
        start_time = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1}/{end_epoch} | Train Acc={train_acc:.4f} | Val Acc={val_acc:.4f} | LR={scheduler.get_last_lr()[0]:.6f} | Time={elapsed:.1f}s")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    print(f"✅ Training complete — Best Val Acc: {best_val_acc:.4f}")
    return best_val_acc

In [ ]:
epochs_1 = 20
train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    start_epoch=0,
    end_epoch=epochs_1,
    save_path="best_vit_model_phase1.pth"
)


🌀 Epoch 1/20


Epoch 1/20 | Train Acc=0.4664 | Val Acc=0.5843 | LR=0.000012 | Time=59.6s

🌀 Epoch 2/20


Epoch 2/20 | Train Acc=0.6947 | Val Acc=0.7326 | LR=0.000012 | Time=23.9s

🌀 Epoch 3/20


Epoch 3/20 | Train Acc=0.8116 | Val Acc=0.8401 | LR=0.000013 | Time=25.6s

🌀 Epoch 4/20


Epoch 4/20 | Train Acc=0.9080 | Val Acc=0.9070 | LR=0.000014 | Time=23.8s

🌀 Epoch 5/20


...

Epoch 17/20 | Train Acc=0.9994 | Val Acc=0.9797 | LR=0.000045 | Time=22.9s

🌀 Epoch 18/20


Epoch 18/20 | Train Acc=0.9994 | Val Acc=0.9738 | LR=0.000049 | Time=24.1s

🌀 Epoch 19/20


Epoch 19/20 | Train Acc=0.9366 | Val Acc=0.9215 | LR=0.000053 | Time=23.9s

🌀 Epoch 20/20


Epoch 20/20 | Train Acc=0.9838 | Val Acc=0.9738 | LR=0.000057 | Time=22.6s
✅ Training complete — Best Val Acc: 0.9913


0.9912790697674418

In [ ]:
from google.colab import files

files.download("best_vit_model_phase1.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ------------------------------
# 🔁 Phase 2 — Resume from saved weights (epoch 21–40)
# ------------------------------
model.load_state_dict(torch.load("best_vit_model_phase1.pth"))
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
epochs_2 = 40
train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    start_epoch=20,
    end_epoch=epochs_2,
    save_path="best_vit_model_phase2.pth"
)


In [ ]:
# ------------------------------
# # 🔁 Phase 3 — Resume from saved weights (epoch 41–60)
# # ------------------------------
# model.load_state_dict(torch.load("best_vit_model_phase2.pth"))
# optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.8)
# epochs_3 = 60
# train_model(
#     model,
#     train_loader,
#     val_loader,
#     optimizer,
#     scheduler,
#     criterion,
#     device,
#     start_epoch=40,
#     end_epoch=epochs_2,
#     save_path="best_vit_model_phase3.pth"
# )